#### Model with Gating at position G (at attention calculation step) :  As in Paper where stated best effective

In [ ]:
### Aplied gated atention at position G elementwise

In [ ]:
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"

In [ ]:
import random
import numpy as np
import torch

SEED = 12

def set_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False

def seed_worker(worker_id):
    worker_seed = torch.initial_seed() % 2**32
    np.random.seed(worker_seed)
    random.seed(worker_seed)

set_seed(SEED)
torch.use_deterministic_algorithms(True)

In [ ]:
# Rope

def compute_rope_params(seq_len, head_dim, device=None):
    # x: (seq_len, dim)

    assert head_dim % 2 == 0, "head_dim must be even for RoPE"

    theta = 1.0 / (10000 ** (torch.arange(0, head_dim, 2).float() / head_dim))

    pos = torch.arange(seq_len).float()

    angles = pos[:, None] * theta[None, :]


    angles = angles[None, None, :, :]

    return torch.cos(angles), torch.sin(angles)



# similar to sebastian
def apply_rope(x, cos, sin, offset=0):

    batch_size, num_heads, seq_len, head_dim = x.shape   # (batch_size, num_heads, seq_len, head_dim)

    assert head_dim % 2 == 0, "Head dimension must be even"

    cos_sel = cos[...,offset : offset + seq_len, :].to(x.device, x.dtype)  #(1,1,seq_len,head_dim//2)
    sin_sel = sin[..., offset : offset + seq_len, :].to(x.device, x.dtype)

    x_even = x[..., 0::2]  # (b,n_heads,seq_len,head_dim//2)
    x_odd  = x[..., 1::2]


    x_rot = torch.empty_like(x)

    x_rot[..., 0::2] = x_even * cos_sel - x_odd * sin_sel
    x_rot[..., 1::2] = x_even * sin_sel + x_odd * cos_sel

    return x_rot.to(dtype=x.dtype)

In [ ]:
import torch
from torch import nn


class causal_multi_head_transformer(nn.Module):
    def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, add_norm, remove_norm, vocab_size, qkv_bias=False):
        super().__init__()
        self.num_heads = num_heads

        assert dout % num_heads == 0, f"d_out must be divisible by num_heads {dout // num_heads}"

        self.head_dim = dout // num_heads

        self.context_length = context_length



        self.wq = nn.Linear(din, dout, bias = qkv_bias)
        self.wk = nn.Linear(din, dout, bias = qkv_bias)
        self.wv = nn.Linear(din, dout, bias = qkv_bias)
        self.dropout = nn.Dropout(dropout)

        self.out_proj = nn.Linear(dout, dout)  # Linear layer to combine head outputs


        # Feedforward
        self.ff = nn.Sequential(nn.Linear(din, ff_dim),
                                            nn.ReLU(), nn.Linear(ff_dim, din),)
        self.norm1 = nn.RMSNorm(din)
        self.norm2 = nn.RMSNorm(din)

        self.apply_rope = apply_rope

        self.gate = nn.Linear(din, self.num_heads * self.head_dim)

        nn.init.zeros_(self.gate.weight)
        nn.init.constant_(self.gate.bias, 2.0)  # sigmoid(2) ≈ 0.88
        # or for even closer to identity
        # nn.init.constant_(self.gate.bias, 4.0)  # sigmoid(4) ≈ 0.98

        self.register_buffer(
            "mask",
            torch.triu(torch.ones(context_length, context_length, dtype=torch.bool), diagonal=1)
        )

        self.qk_norm  = qk_norm
        self.pre_norm = pre_norm
        self.post_norm = post_norm

        self.remove_norm = remove_norm
        self.add_norm = add_norm


        if self.qk_norm:
            self.q_norm = nn.RMSNorm(self.head_dim)
            self.k_norm = nn.RMSNorm(self.head_dim)

        if (self.pre_norm and self.post_norm) or self.add_norm:
            self.post_attn_norm = nn.RMSNorm(dout)

        if self.pre_norm and self.post_norm:
          self.post_ff_norm = nn.RMSNorm(dout)
          self.pre_ff_norm = nn.RMSNorm(dout)

    def forward(self, x, cos, sin, start_pos):

        b, num_tokens, d_in = x.shape

        assert num_tokens <= self.context_length

        #Pre-LN
        if self.pre_norm:
          x_norm = self.norm1(x)

        else:
          x_norm =x

        q = self.wq(x_norm)
        k = self.wk(x_norm)
        v = self.wv(x_norm)

        k = k.view(b, num_tokens, self.num_heads, self.head_dim)
        v = v.view(b, num_tokens, self.num_heads, self.head_dim)
        q = q.view(b, num_tokens, self.num_heads, self.head_dim)


        k = k.transpose(1,2)  # reshape to (b, num_heads, num_tokens, head_dim)
        v = v.transpose(1,2)
        q = q.transpose(1,2)

        #QK normalization (optional)
        if self.qk_norm:
            q = self.q_norm(q)
            k = self.k_norm(k)

        q_enc = self.apply_rope(q, cos, sin, start_pos)
        k_enc = self.apply_rope(k, cos, sin, start_pos)

        attn_scores = q_enc @ k_enc.transpose(2,3)    # (b, num_heads, num_tokens, head_dim) @ (b, num_heads, head_dim, num_tokens)

        attn_scores = attn_scores / (self.head_dim ** 0.5)

        mask = self.mask[:num_tokens, :num_tokens]
        masked = attn_scores.masked_fill(mask.bool(), float('-inf'))

        attn_weights = torch.softmax(masked, dim=-1)                             # shape: (b, num_heads, num_tokens, num_tokens)
        attn_weights = self.dropout(attn_weights)
        z = (attn_weights @ v)

        #gate applied elemnetwise

        gate_val = self.gate(x_norm)

        gate_val = gate_val.view(b, num_tokens, self.num_heads, self.head_dim)
        gate_val = gate_val.transpose(1,2)


        z = z * torch.sigmoid(gate_val)

       # Reshape and project
        z = z.transpose(1,2).contiguous().view(b, num_tokens, -1)  # Reshape to (b, num_tokens, num_heads * head_dim)
        attn_out = self.out_proj(z)

        # after attention output
        if self.pre_norm and self.post_norm and self.remove_norm:
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out

        elif self.pre_norm and self.add_norm:
          attn_out = self.post_attn_norm(attn_out)
          x= x + attn_out
          x = x + self.ff(self.norm2(x))


        elif self.pre_norm and self.post_norm:
          attn_out = self.post_attn_norm(attn_out)
          x = x + attn_out

          ff_out = self.ff(self.pre_ff_norm(x))
          ff_out = self.post_ff_norm(ff_out)
          x = x + ff_out

        elif self.pre_norm:                       # Pre-LN
          x= x + attn_out
          x = x + self.ff(self.norm2(x))


        elif self.post_norm:                    # Post-LN
          x = self.norm1(x + attn_out)
          x = self.norm2(x + self.ff(x))

        sink_val = attn_weights[:, :, :, 0].mean(dim=-1)

        return x, gate_val, sink_val

In [ ]:
class Gated_Transformer_LM(nn.Module):
  def __init__(self, din, dout, context_length, dropout, ff_dim, num_heads, vocab_size, qk_norm, pre_norm, post_norm, add_norm, remove_norm, n_transformer):
    super().__init__()


    assert din == dout

    self.embedding = nn.Embedding(vocab_size, din)


    self.dstack = nn.ModuleList([causal_multi_head_transformer(din, dout, context_length, dropout, ff_dim, num_heads, qk_norm, pre_norm, post_norm, add_norm, remove_norm, vocab_size)
    for _ in range(n_transformer)])

    cos, sin = compute_rope_params(context_length, din//num_heads)

    # register so they move with the model and are saved in state_dict
    self.register_buffer("rope_cos", cos)
    self.register_buffer("rope_sin", sin)

    self.cos, self.sin = cos, sin

    self.final_norm = nn.RMSNorm(din)
    self.out_head = nn.Linear(
        din, vocab_size, bias=False
    )


  def forward(self, inp, start_pos: int = 0):
      gate_vals = []
      attn_sink_val = []

      x = self.embedding(inp)

      for layer in self.dstack:
        x, gate_val, sink_info = layer(x, self.cos, self.sin, start_pos)

        gate_vals.append(gate_val)
        attn_sink_val.append(sink_info)

      x = self.final_norm(x)
      logits = self.out_head(x)


      return logits, gate_vals, attn_sink_val # Only return logits, as gate_val is not computed

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader, Subset
from datasets import load_dataset
from transformers import AutoTokenizer


# -----------------------------
# Load Dataset
# -----------------------------
# dataset = load_dataset("Salesforce/wikitext", "wikitext-2-raw-v1")
dataset = load_dataset("Salesforce/wikitext", "wikitext-103-raw-v1")

texts = dataset["train"]["text"]
texts = [t for t in texts if len(t.strip()) > 0]


# -----------------------------
# Tokenizer (FAST)
# -----------------------------
tokenizer = AutoTokenizer.from_pretrained("gpt2", use_fast=True)
tokenizer.pad_token = tokenizer.eos_token


# -----------------------------
# FAST Tokenization (BATCHED)
# -----------------------------
encodings = tokenizer(
    texts,
    padding=False,
    truncation=False
)

# Flatten tokens + add EOS between docs
all_tokens = [
    token
    for ids in encodings["input_ids"]
    for token in (ids + [tokenizer.eos_token_id])
]

tokens = torch.tensor(all_tokens, dtype=torch.long)


# -----------------------------
# Lazy Dataset (NO stacking)
# -----------------------------
class WikiTextDataset(Dataset):

    def __init__(self, tokens, seq_len=64, stride=None):
        self.tokens = tokens
        self.seq_len = seq_len
        self.stride = stride if stride is not None else seq_len

        self.num_samples = (len(tokens) - (seq_len + 1)) // self.stride

    def __len__(self):
        return self.num_samples

    def __getitem__(self, idx):
        i = idx * self.stride
        chunk = self.tokens[i : i + self.seq_len + 1]

        x = chunk[:-1]
        y = chunk[1:]

        return x, y


# -----------------------------
# Create Dataset
# -----------------------------
seq_len = 128

train_dataset = WikiTextDataset(
    tokens,
    seq_len=seq_len,
    stride=128
)

max_samples = 300000
train_dataset = Subset(train_dataset, range(min(max_samples, len(train_dataset))))


# # -----------------------------
# # DataLoader (OPTIMIZED)
# # -----------------------------
# dataloader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )


# # -----------------------------
# # Example Batch
# # -----------------------------
# for x, y in dataloader:
#     print("Input shape:", x.shape)
#     print("Target shape:", y.shape)
#     break


README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

wikitext-103-raw-v1/test-00000-of-00001.(…):   0%|          | 0.00/733k [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00000-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/train-00001-of-00002(…):   0%|          | 0.00/157M [00:00<?, ?B/s]

wikitext-103-raw-v1/validation-00000-of-(…):   0%|          | 0.00/657k [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/1.04M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

[transformers] Token indices sequence length is longer than the specified maximum sequence length for this model (1063 > 1024). Running this sequence through the model will result in indexing errors


In [ ]:
# dataloader = DataLoader(
#     train_dataset,
#     batch_size=16,
#     shuffle=True,
#     num_workers=2,
#     pin_memory=True
# )


# len(dataloader)

In [ ]:
vocab_size = tokenizer.vocab_size
print(vocab_size)

50257


In [ ]:
outputs = tokenizer("a gaoal is good",  return_tensors="pt", add_special_tokens=True)['input_ids']
print(outputs)
print(tokenizer.decode(outputs, skip_special_tokens=False))

tensor([[  64,  308, 5488,  282,  318,  922]])
['a gaoal is good']


### Training with Gating

In [ ]:
import math
import copy
from torch.utils.data import DataLoader, random_split

def model_training_with_gate(dataloader_dataset, qk_norm, pre_norm, post_norm,
                             add_norm, remove_norm,
                             vocab_size, n_transformer, seed,
                             val_ratio=0.2, patience=10):

    device = "cuda" if torch.cuda.is_available() else "cpu"

    set_seed(seed)

    gen = torch.Generator()
    gen.manual_seed(seed)

    val_size = int(len(dataloader_dataset) * val_ratio)
    train_size = len(dataloader_dataset) - val_size
    split_gen = torch.Generator().manual_seed(seed)
    train_dataset, val_dataset = random_split(
      dataloader_dataset, [train_size, val_size], generator=split_gen
    )

    train_dataloader = DataLoader(
      train_dataset, batch_size=16, shuffle=True,
      num_workers=0,
      pin_memory=True,
      generator=gen, worker_init_fn=seed_worker
    )

    print('train_dataloader size: ', len(train_dataloader))

    val_dataloader = DataLoader(
      val_dataset, batch_size=16, shuffle=False,
      num_workers=0, pin_memory=True
    )

    print('val_dataloader size: ', len(val_dataloader))


    model = Gated_Transformer_LM(
      din=256, dout=256, context_length=128, dropout=0.1,
      ff_dim=1024, num_heads=8,
      vocab_size=vocab_size, qk_norm=qk_norm,
      pre_norm=pre_norm, post_norm=post_norm,
      add_norm = add_norm, remove_norm = remove_norm,
      n_transformer=n_transformer
    ).to(device)

    optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)
    loss_fn = nn.CrossEntropyLoss()

    loss_history, gate_mean_history, max_act_history, grad_norm_history = [], [], [], []

    print("\n--- Training with Gating ---")
    gated_epoch_loss = []

    train_ppl_history = []
    val_ppl_history = []
    val_epoch_loss = []

    best_val_ppl = float("inf")
    best_model_state = copy.deepcopy(model.state_dict())
    epochs_without_improvement = 0

    for epoch in range(10):
        model.train()
        total_loss = 0

        for step, (xb, yb) in enumerate(train_dataloader):
            xb, yb = xb.to(device), yb.to(device)
            optimizer.zero_grad()
            out, gate_vals, attn_sink_info = model(xb)
            loss = loss_fn(out.view(-1, vocab_size), yb.view(-1))
            loss.backward()

            total_norm = torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
            optimizer.step()

            # Existing statistics (UNCHANGED)
            gates = [torch.sigmoid(g) for g in gate_vals]
            layer_means = [g.mean().item() for g in gates]
            layer_std = [g.std().item() for g in gates]
            layer_sparsity = [(g < 0.1).float().mean().item() for g in gates]

            all_gates_combined = torch.stack(gate_vals)
            g = torch.sigmoid(all_gates_combined)

            gate_mean = g.mean().item()
            sparsity = (g < 0.1).float().mean()
            head_mean = g.mean(dim=(0, 1, 3, 4))

            dead = torch.mean(torch.stack([
              (g < 0.01).float().mean() for g in gates
            ])).item()

            open_ = torch.mean(torch.stack([
              (g > 0.99).float().mean() for g in gates
            ])).item()

            loss_history.append(loss.item())
            gate_mean_history.append(gate_mean)
            max_act_history.append(out.abs().max().item())
            grad_norm_history.append(total_norm.item())
            total_loss += loss.item()

            if step % 500 == 0:
              out_mean = out.mean().item()
              sink_val_tensor = attn_sink_info[-1]
              avg_sink_val = sink_val_tensor.float().mean().item()

              print(
                  f"Epoch {epoch} Step {step} | Loss={loss.item():.4f} | "
                  f"gate_mean={gate_mean:.4f} | "
                  f"LayerMeans={[round(m, 3) for m in layer_means]} | "
                  f"LayerStd={[round(s, 3) for s in layer_std]} | "
                  f"LayerSparsity={[round(s, 3) for s in layer_sparsity]} | "
                  f"MaxAct={out.abs().max():.4f} | MeanAct={out_mean:.4f} | GradNorm={total_norm:.4f} | "
                  f"Sparsity={sparsity.item():.4f} | HeadMeanAvg={head_mean.mean().item():.4f} | "
                  f"Sink Val={avg_sink_val:.4f} | Dead={dead:.3f} | Open={open_:.3f}"
              )

        avg_train_loss = total_loss / len(train_dataloader)
        train_ppl = math.exp(min(avg_train_loss, 20))
        train_ppl_history.append(train_ppl)

        gated_epoch_loss.append(avg_train_loss)

        model.eval()
        total_val_loss = 0.0

        with torch.no_grad():
            for xb, yb in val_dataloader:
                xb, yb = xb.to(device), yb.to(device)
                val_out, _, _ = model(xb)
                val_loss = loss_fn(val_out.view(-1, vocab_size), yb.view(-1))
                total_val_loss += val_loss.item()

        avg_val_loss = total_val_loss / len(val_dataloader)
        val_ppl = math.exp(min(avg_val_loss, 20))

        val_epoch_loss.append(avg_val_loss)
        val_ppl_history.append(val_ppl)

        print(
            f"\nEpoch {epoch} Summary | "
            f"Train Loss={avg_train_loss:.4f} | Train PPL={train_ppl:.2f} | "
            f"Val Loss={avg_val_loss:.4f} | Val PPL={val_ppl:.2f}\n"
        )

        if val_ppl < best_val_ppl:
            best_val_ppl = val_ppl
            best_model_state = copy.deepcopy(model.state_dict())
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1
            if epochs_without_improvement >= patience:
                print(f"Early stopping triggered at epoch {epoch}.")
                break

    model.load_state_dict(best_model_state)

    return {
    "model": model,
    "train_loss": gated_epoch_loss,
    "val_loss": val_epoch_loss,
    "train_ppl": train_ppl_history,
    "val_ppl": val_ppl_history,
    "step_loss": loss_history,
    "gate_mean_history": gate_mean_history,
    "max_activation": max_act_history,
    "grad_norm": grad_norm_history,
    }

### config 2+  : pre norm =True, post_norm = False, qk norm = True, add_norm = True, remove_norm = False

**Label note:** the seed-42/100 log cells below print "Config 1 (With Gate)" — leftover
banner text. The training calls use the C2+ flags (`post_norm=False, add_norm=True`),
and the results match paper Table 3. The bottom summary labels these rows "Config 1 (2+)".

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = True, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9725 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3302 | MeanAct=0.0006 | GradNorm=2.3801 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0451 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0654 | gate_mean=0.7142 | LayerMeans=[0.831, 0.661, 0.69, 0.676, 0.684, 0.637, 0.738, 0.795] | LayerStd=[0.141, 0.27, 0.268, 0.278, 0.276, 0.301, 0.245, 0.199] | LayerSparsity=[0.003, 0.027, 0.03, 0.041, 0.032, 0.069, 0.019, 0.005] | MaxAct=15.5141 | MeanAct=-3.7931 | GradNorm=0.5469 | Sparsity=0.0283 | HeadMeanAvg=0.7142 | Sink Val=0.0404 | Dead=0.001 | Open=0.013
Epoch 0 Step 1000 | Loss=5.6992 | gate_mean=0.5742 | LayerMeans=[0.806, 0.522, 0.558, 0.487, 0.521, 0.46, 0.536, 0.704] | LayerStd=[0.181, 0.329, 0.33, 0.338

In [ ]:
print(f'####Logs for Config 2+ (With Gate) - SEED {SEED}')
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

####Logs for Config 2+ (With Gate) - SEED 12
train_loss:  [4.706863799858093, 4.046785996437073, 3.8510110047976176, 3.7367382993380227, 3.657713229942322, 3.5974653097629545, 3.5494415802955626, 3.5092356625239054, 3.4751602325757345, 3.44504256122907]
val_loss_no_gate:  [4.212993700853984, 3.986436112467448, 3.880072175280253, 3.8211403666178385, 3.7847640102386473, 3.7575386294047037, 3.739415282821655, 3.72517106145223, 3.7179760703404745, 3.7038029495875042]
train_ppl_no_gate:  [110.70442367627042, 57.21327754932261, 47.04059746858314, 41.96090281309685, 38.77257744779137, 36.505586722380286, 34.79388247209295, 33.422711787314356, 32.30300411550483, 31.344617483724882]
val_ppl_no_gate:  [67.55848727519165, 53.862586693136564, 48.427710228769456, 45.65624350160329, 44.025279817466256, 42.84284397746568, 42.07338194210008, 41.47832748093048, 41.18096233599167, 40.60141627368236]


Best Training PPL:  31.344617483724882
Best Validation PPL:  40.60141627368236


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = True, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9911 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.1616 | MeanAct=-0.0021 | GradNorm=2.2341 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0476 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9222 | gate_mean=0.7172 | LayerMeans=[0.828, 0.623, 0.673, 0.638, 0.728, 0.784, 0.752, 0.711] | LayerStd=[0.146, 0.282, 0.285, 0.299, 0.256, 0.213, 0.237, 0.27] | LayerSparsity=[0.005, 0.043, 0.046, 0.062, 0.026, 0.01, 0.013, 0.034] | MaxAct=15.6498 | MeanAct=-3.8886 | GradNorm=0.5394 | Sparsity=0.0299 | HeadMeanAvg=0.7172 | Sink Val=0.0329 | Dead=0.001 | Open=0.013
Epoch 0 Step 1000 | Loss=5.8122 | gate_mean=0.5857 | LayerMeans=[0.806, 0.471, 0.568, 0.465, 0.582, 0.659, 0.596, 0.54] | LayerStd=[0.178, 0.326, 0.335, 0.

In [ ]:
print('####Logs for Config 1 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

####Logs for Config 1 (With Gate) - SEED 42
train_loss:  [4.71285870642662, 4.048257037146886, 3.8491895418167115, 3.7330187095801035, 3.6521251460234323, 3.5908373668988545, 3.5419729616641997, 3.500735712766647, 3.4660996017615, 3.435664934094747]
val_loss:  [4.210004608472189, 3.980541583251953, 3.875533679262797, 3.8182167496999107, 3.7759819287618, 3.7514706084569296, 3.731033896827698, 3.7177906951268516, 3.7063413670857748, 3.700536572329203]
train_ppl:  [111.37007963198532, 57.29750254374898, 46.95499274827668, 41.80511538104295, 38.55651727468745, 36.2644298491793, 34.53498822782391, 33.13982438392447, 32.01164048346431, 31.052053271345066]
val_ppl:  [67.35685022157229, 53.54602600694543, 48.20841926076597, 45.522957069865555, 43.64033898797462, 42.58365986534078, 41.722222345708644, 41.173329113828395, 40.7046105388587, 40.469013088008374]


Best Training PPL:  31.052053271345066
Best Validation PPL:  40.469013088008374


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = True, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9822 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.5814 | MeanAct=0.0001 | GradNorm=2.2615 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0470 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0767 | gate_mean=0.7243 | LayerMeans=[0.828, 0.683, 0.629, 0.671, 0.703, 0.736, 0.77, 0.776] | LayerStd=[0.146, 0.26, 0.31, 0.287, 0.273, 0.248, 0.224, 0.222] | LayerSparsity=[0.005, 0.025, 0.077, 0.05, 0.035, 0.021, 0.013, 0.009] | MaxAct=15.7233 | MeanAct=-3.6215 | GradNorm=0.5230 | Sparsity=0.0294 | HeadMeanAvg=0.7243 | Sink Val=0.0389 | Dead=0.001 | Open=0.013
Epoch 0 Step 1000 | Loss=5.6074 | gate_mean=0.5992 | LayerMeans=[0.801, 0.515, 0.493, 0.532, 0.545, 0.584, 0.641, 0.682] | LayerStd=[0.185, 0.324, 0.345, 0.33

In [ ]:
print('####Logs for Config 1 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

####Logs for Config 1 (With Gate) - SEED 100
train_loss:  [4.7042211116313934, 4.045677745501201, 3.846375035206477, 3.7300976766268414, 3.6490928129514058, 3.5875451395193734, 3.5385098558266956, 3.497681748755773, 3.4631079634984334, 3.432809568532308]
val_loss:  [4.213103398895264, 3.986785315831502, 3.8806590406417847, 3.8237746481577557, 3.7818941310246785, 3.7531052741368613, 3.7349893287658693, 3.72135444577535, 3.7089129375457763, 3.70079794921875]
train_ppl:  [110.41225262771138, 57.14990600320808, 46.82302341220317, 41.68317943755541, 38.439778157607776, 36.14523541557743, 34.4155967604606, 33.0387709382636, 31.91601634266855, 30.963514772836202]
val_ppl:  [67.56589871542039, 53.88139897407118, 48.456139115589544, 45.776673454599525, 43.899113709368685, 42.653326838240076, 41.88757856855966, 41.32032236022549, 40.80942001762754, 40.479592135271346]


Best Training PPL:  30.963514772836202
Best Validation PPL:  40.479592135271346


### config 2: pre norm =True, post_norm = False, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9605 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.3866 | MeanAct=-0.0002 | GradNorm=1.7546 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0422 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.8988 | gate_mean=0.5842 | LayerMeans=[0.84, 0.497, 0.521, 0.527, 0.536, 0.576, 0.584, 0.591] | LayerStd=[0.115, 0.247, 0.27, 0.285, 0.288, 0.295, 0.303, 0.292] | LayerSparsity=[0.001, 0.041, 0.059, 0.078, 0.081, 0.074, 0.081, 0.065] | MaxAct=15.7915 | MeanAct=-3.8140 | GradNorm=0.5501 | Sparsity=0.0599 | HeadMeanAvg=0.5842 | Sink Val=0.0319 | Dead=0.001 | Open=0.003
Epoch 0 Step 1000 | Loss=5.5493 | gate_mean=0.4285 | LayerMeans=[0.812, 0.313, 0.334, 0.353, 0.346, 0.403, 0.419, 0.447] | LayerStd=[0.151, 0.267, 0.283, 0

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 12')
print('train_loss: ', model_det['train_loss'])
print('val_loss: ', model_det['val_loss'])
print('train_ppl: ', model_det['train_ppl'])
print('val_ppl: ', model_det['val_ppl'])
print('\nBest Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 12
train_loss:  [4.677497733640671, 4.054813833030065, 3.8610034872214, 3.7481135477224985, 3.669686558834712, 3.610477603260676, 3.5632950085163118, 3.52421307776769, 3.490927352841695, 3.462142832740148]
val_loss:  [4.213144381904602, 3.988619053586324, 3.884658376757304, 3.822900394821167, 3.783943663978577, 3.758944452857971, 3.740096597099304, 3.725758367093404, 3.7143018840789797, 3.702404843711853]
train_ppl:  [107.5007402553912, 57.67442492372607, 47.51300614889385, 42.44094362046201, 39.23960462514932, 36.98371213892089, 35.279251273831, 33.927065147040466, 32.81636595004742, 31.885228073357034]
val_ppl:  [67.56866782602124, 53.980293975689705, 48.650319540344746, 45.73667053398524, 43.989178653571415, 42.903115808184005, 42.10205690514786, 41.502695091563666, 41.02993343400303, 40.544690858338704]

Best Training PPL:  31.885228073357034
Best Validation PPL:  40.544690858338704


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0048 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.4270 | MeanAct=-0.0005 | GradNorm=1.5469 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0429 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.7823 | gate_mean=0.5538 | LayerMeans=[0.838, 0.504, 0.519, 0.486, 0.519, 0.504, 0.522, 0.538] | LayerStd=[0.116, 0.243, 0.273, 0.294, 0.304, 0.297, 0.294, 0.307] | LayerSparsity=[0.001, 0.04, 0.064, 0.117, 0.113, 0.114, 0.094, 0.105] | MaxAct=15.8534 | MeanAct=-3.8639 | GradNorm=0.5471 | Sparsity=0.0811 | HeadMeanAvg=0.5538 | Sink Val=0.0281 | Dead=0.003 | Open=0.002
Epoch 0 Step 1000 | Loss=5.6645 | gate_mean=0.4195 | LayerMeans=[0.812, 0.29, 0.353, 0.321, 0.38, 0.356, 0.403, 0.44] | LayerStd=[0.149, 0.258, 0.293, 0.3

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])
print('\nBest Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 42
train_loss:  [4.674583984629313, 4.048083135430018, 3.8540851747035982, 3.7411240969022117, 3.662707894484202, 3.6038171689828236, 3.556957592169444, 3.5177689365228018, 3.4846543903191884, 3.455996220334371]
val_loss:  [4.202537799708049, 3.9783079634984335, 3.8743236846923828, 3.816750860595703, 3.777504063288371, 3.7509503920237224, 3.730872834587097, 3.7187025641759237, 3.7045137314478556, 3.6991321554819745]
train_ppl:  [107.18796597391461, 57.28753927602419, 47.18543076469807, 42.145338992680166, 38.96671789517514, 36.73820306100053, 35.05637893344138, 33.70913728034174, 32.61115443132431, 31.689843028142324]
val_ppl:  [66.85578251838166, 53.42655801823298, 48.15012261163926, 45.45627434978275, 43.70681603536899, 42.56151290679807, 41.71550301222477, 41.2108909214167, 40.630285282442735, 40.41221761583106]

Best Training PPL:  31.689843028142324
Best Validation PPL:  40.41221761583106


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = False, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0359 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.4045 | MeanAct=-0.0004 | GradNorm=1.5553 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0434 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9413 | gate_mean=0.5818 | LayerMeans=[0.838, 0.498, 0.541, 0.588, 0.525, 0.549, 0.547, 0.568] | LayerStd=[0.115, 0.241, 0.279, 0.293, 0.292, 0.289, 0.295, 0.29] | LayerSparsity=[0.001, 0.037, 0.066, 0.068, 0.091, 0.072, 0.087, 0.07] | MaxAct=16.0255 | MeanAct=-3.6054 | GradNorm=0.5208 | Sparsity=0.0615 | HeadMeanAvg=0.5818 | Sink Val=0.0331 | Dead=0.001 | Open=0.003
Epoch 0 Step 1000 | Loss=5.4710 | gate_mean=0.4317 | LayerMeans=[0.81, 0.302, 0.368, 0.405, 0.358, 0.375, 0.39, 0.445] | LayerStd=[0.154, 0.261, 0.296, 0.3

In [ ]:
print('#### Logs for Config 2 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])
print('\nBest Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

#### Logs for Config 2 (With Gate) - SEED 100
train_loss:  [4.672028570572535, 4.0468161436875665, 3.8508571123600004, 3.7370175007184345, 3.6582455904483795, 3.598961913061142, 3.5517559236049654, 3.51274056353569, 3.4794756201903025, 3.4504711392084757]
val_loss:  [4.2063358335495, 3.9837602863311767, 3.8788534534454344, 3.8217034233729046, 3.780849390411377, 3.75204243850708, 3.73317264251709, 3.7206431910832722, 3.708281772295634, 3.701049018796285]
train_ppl:  [106.91440601729207, 57.21500239833203, 47.03335883337162, 41.97261999073556, 38.7932239319414, 36.560262007271874, 34.874500714361346, 33.540060611551404, 32.44270531499167, 31.515236875301028]
val_ppl:  [67.11018585330565, 53.71865243330438, 48.368726270722824, 45.68195779654219, 43.85327447161464, 42.6080174452315, 41.81155106055343, 41.290943536220354, 40.783670656739666, 40.48975660530874]

Best Training PPL:  31.515236875301028
Best Validation PPL:  40.48975660530874


### config 3-  : pre norm = True, post_norm = True, qk norm = True, remove_norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9986 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.0970 | MeanAct=0.0005 | GradNorm=3.4152 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0422 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9369 | gate_mean=0.6573 | LayerMeans=[0.844, 0.439, 0.61, 0.659, 0.671, 0.682, 0.675, 0.679] | LayerStd=[0.106, 0.262, 0.288, 0.279, 0.281, 0.295, 0.292, 0.281] | LayerSparsity=[0.0, 0.089, 0.057, 0.043, 0.046, 0.058, 0.053, 0.044] | MaxAct=15.8146 | MeanAct=-3.7969 | GradNorm=0.5658 | Sparsity=0.0486 | HeadMeanAvg=0.6573 | Sink Val=0.0347 | Dead=0.001 | Open=0.013
Epoch 0 Step 1000 | Loss=5.5871 | gate_mean=0.4959 | LayerMeans=[0.816, 0.277, 0.409, 0.45, 0.459, 0.489, 0.515, 0.553] | LayerStd=[0.141, 0.268, 0.331, 0.33

In [ ]:
print('#### Logs for Config 3- (With Gate) - SEED 12')
print('train_loss: ', model_det['train_loss'])
print('val_loss: ', model_det['val_loss'])
print('train_ppl: ', model_det['train_ppl'])
print('val_ppl: ', model_det['val_ppl'])
print('\nBest Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

#### Logs for Config 3- (With Gate) - SEED 12
train_loss:  [4.68646365017891, 4.04484661008517, 3.847583936214447, 3.732665313529968, 3.652726853513718, 3.591547238270442, 3.542570856555303, 3.50156455133756, 3.4664713250319164, 3.4356535098552703]
val_loss:  [4.205420674260457, 3.9767026623408, 3.8692292346954344, 3.810412886238098, 3.771874342918396, 3.746582894897461, 3.726938988049825, 3.709348044459025, 3.7003044654846193, 3.6891630737304686]
train_ppl:  [108.46891672927146, 57.10242642601822, 46.87966204075212, 41.79034422857904, 38.57972400106132, 36.290182069036575, 34.5556426948345, 33.16730333484124, 32.02354216708555, 31.0516985272786]
val_ppl:  [67.04879743770651, 53.340861105877586, 47.90544799013811, 45.16908471021198, 43.46145020139137, 42.37603096225757, 41.55172297911761, 40.827180341942636, 40.45962104309821, 40.011346393615135]

Best Training PPL:  31.0516985272786
Best Validation PPL:  40.011346393615135


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9950 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.2516 | MeanAct=0.0009 | GradNorm=3.0835 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0439 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.7781 | gate_mean=0.6525 | LayerMeans=[0.841, 0.489, 0.605, 0.643, 0.66, 0.683, 0.647, 0.652] | LayerStd=[0.111, 0.266, 0.282, 0.289, 0.298, 0.267, 0.291, 0.298] | LayerSparsity=[0.001, 0.066, 0.053, 0.057, 0.068, 0.038, 0.056, 0.061] | MaxAct=15.9730 | MeanAct=-3.9105 | GradNorm=0.5954 | Sparsity=0.0499 | HeadMeanAvg=0.6525 | Sink Val=0.0280 | Dead=0.001 | Open=0.010
Epoch 0 Step 1000 | Loss=5.6916 | gate_mean=0.4913 | LayerMeans=[0.815, 0.288, 0.43, 0.451, 0.449, 0.474, 0.494, 0.529] | LayerStd=[0.144, 0.28, 0.328, 0.3

In [ ]:
print('#### Logs for Config 3- (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])
print('\nBest Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

#### Logs for Config 3- (With Gate) - SEED 42
train_loss:  [4.688890517075857, 4.0521198081970216, 3.8534164527734123, 3.7368161187966664, 3.655524889262517, 3.5937794095516207, 3.544097486337026, 3.502453117609024, 3.467140355571111, 3.436087728691101]
val_loss:  [4.2053450566609705, 3.9794401291529335, 3.8741925715128582, 3.8145516799290973, 3.7750299357096355, 3.7458479331334433, 3.7272559052149457, 3.710787375831604, 3.700718466250102, 3.6911231696446736]
train_ppl:  [108.73247603494987, 57.51925769677873, 47.15388738041234, 41.964168314896106, 38.687822609354946, 36.37127844807148, 34.608436656262455, 33.19678777940152, 32.044974063265634, 31.065184687424157]
val_ppl:  [67.04372756028418, 53.48707998624138, 48.1438099098174, 45.356417631784154, 43.598813457608344, 42.34489764209087, 41.56489352024507, 40.885986494080285, 40.47637482498115, 40.08984938176238]

Best Training PPL:  31.065184687424157
Best Validation PPL:  40.08984938176238


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = True, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0627 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.2458 | MeanAct=-0.0012 | GradNorm=2.9443 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0447 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.9444 | gate_mean=0.6644 | LayerMeans=[0.841, 0.483, 0.628, 0.664, 0.672, 0.665, 0.697, 0.665] | LayerStd=[0.111, 0.264, 0.282, 0.284, 0.274, 0.292, 0.277, 0.29] | LayerSparsity=[0.001, 0.074, 0.049, 0.05, 0.039, 0.057, 0.044, 0.053] | MaxAct=16.2022 | MeanAct=-3.6002 | GradNorm=0.5699 | Sparsity=0.0458 | HeadMeanAvg=0.6644 | Sink Val=0.0274 | Dead=0.002 | Open=0.013
Epoch 0 Step 1000 | Loss=5.4987 | gate_mean=0.4863 | LayerMeans=[0.814, 0.287, 0.418, 0.446, 0.429, 0.469, 0.508, 0.52] | LayerStd=[0.149, 0.275, 0.322, 0.

In [ ]:
print('#### Logs for Config 3- (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])
print('\nBest Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

#### Logs for Config 3- (With Gate) - SEED 100
train_loss:  [4.68432370985349, 4.042948279269536, 3.846317842944463, 3.7315008697986602, 3.651108708349864, 3.5901761433124544, 3.541050448449453, 3.5002732965628307, 3.4652986118475595, 3.4347962914148966]
val_loss:  [4.201138925298055, 3.975912160174052, 3.874048917897542, 3.81769077650706, 3.7773471578598024, 3.746094407081604, 3.7279718249638876, 3.7135845273335777, 3.70232744325002, 3.6958900927861533]
train_ppl:  [108.23704790151916, 56.99412995407321, 46.82034557415644, 41.741710045592995, 38.51734688846838, 36.24045887877456, 34.50314393546125, 33.12450353471518, 31.98600974865876, 31.025091844195046]
val_ppl:  [66.76232505792173, 53.298711701389884, 48.1368943742015, 45.4990195105966, 43.699958736655354, 42.35533584251656, 41.59466130277915, 41.00051088898306, 40.54155280198648, 40.281410829120645]

Best Training PPL:  31.025091844195046
Best Validation PPL:  40.281410829120645


### config 3: pre norm = True, post_norm = True, qk norm = True

In [ ]:
model_det = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9528 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.2551 | MeanAct=0.0005 | GradNorm=3.6987 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0443 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0016 | gate_mean=0.7299 | LayerMeans=[0.839, 0.652, 0.696, 0.717, 0.748, 0.69, 0.75, 0.747] | LayerStd=[0.128, 0.267, 0.275, 0.266, 0.242, 0.286, 0.241, 0.253] | LayerSparsity=[0.001, 0.029, 0.037, 0.032, 0.016, 0.045, 0.015, 0.021] | MaxAct=15.8380 | MeanAct=-3.7533 | GradNorm=0.6440 | Sparsity=0.0244 | HeadMeanAvg=0.7299 | Sink Val=0.0343 | Dead=0.000 | Open=0.014
Epoch 0 Step 1000 | Loss=5.6107 | gate_mean=0.6136 | LayerMeans=[0.82, 0.504, 0.563, 0.572, 0.621, 0.559, 0.617, 0.651] | LayerStd=[0.16, 0.316, 0.33, 0.336

In [ ]:
print(f'#### Logs for Config 3 (With Gate) - SEED {SEED}')
print('train_loss: ', model_det['train_loss'])
print('val_loss_no_gate: ', model_det['val_loss'])
print('train_ppl_no_gate: ', model_det['train_ppl'])
print('val_ppl_no_gate: ', model_det['val_ppl'])

print('\n')

print('Best Training PPL: ', min(model_det['train_ppl']))
print('Best Validation PPL: ', min(model_det['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 12
train_loss:  [4.693947413667043, 4.039166353750229, 3.839146086661021, 3.7214957962989805, 3.639353790807724, 3.576455140765508, 3.5259627953211465, 3.483648306798935, 3.447443167289098, 3.4158053368409473]
val_loss_no_gate:  [4.205860771814982, 3.97514150651296, 3.8727877668380737, 3.80864072303772, 3.769423676363627, 3.7443810899098713, 3.7228513161977133, 3.7109914608637493, 3.7012292214075724, 3.689115228907267]
train_ppl_no_gate:  [109.28371753328477, 56.778989478605126, 46.48576267417898, 41.32616342223656, 38.06722938195846, 35.74659932236434, 33.98647989281907, 32.5783613853122, 31.41995395281793, 30.441455183898167]
val_ppl_no_gate:  [67.07831194364724, 53.257652677261454, 48.076224743687625, 45.089108606778325, 43.35507108207666, 42.28282984892153, 41.38221984330746, 40.89433156347114, 40.49705362263046, 40.00943210361568]


Best Training PPL:  30.441455183898167
Best Validation PPL:  40.00943210361568


In [ ]:
SEED2 = 42
model_det2 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED2)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=10.9753 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.5888 | MeanAct=-0.0024 | GradNorm=3.3174 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0467 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=5.8460 | gate_mean=0.7238 | LayerMeans=[0.838, 0.662, 0.667, 0.758, 0.721, 0.707, 0.748, 0.688] | LayerStd=[0.131, 0.267, 0.294, 0.237, 0.266, 0.274, 0.248, 0.289] | LayerSparsity=[0.003, 0.032, 0.06, 0.017, 0.034, 0.033, 0.021, 0.046] | MaxAct=16.1630 | MeanAct=-3.9300 | GradNorm=0.6038 | Sparsity=0.0307 | HeadMeanAvg=0.7238 | Sink Val=0.0293 | Dead=0.001 | Open=0.016
Epoch 0 Step 1000 | Loss=5.7720 | gate_mean=0.6196 | LayerMeans=[0.822, 0.527, 0.554, 0.679, 0.615, 0.57, 0.632, 0.558] | LayerStd=[0.157, 0.318, 0.34, 0.

In [ ]:
print('#### Logs for Config 3 (With Gate) - SEED 42')
print('train_loss: ', model_det2['train_loss'])
print('val_loss: ', model_det2['val_loss'])
print('train_ppl: ', model_det2['train_ppl'])
print('val_ppl: ', model_det2['val_ppl'])
print('\nBest Training PPL: ', min(model_det2['train_ppl']))
print('Best Validation PPL: ', min(model_det2['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 42
train_loss:  [4.6967745681285855, 4.041444413312276, 3.8421232211589813, 3.725096893453598, 3.64279905722936, 3.57989799434344, 3.52950784184138, 3.486871440633138, 3.4505379260381064, 3.4185952102820076]
val_loss:  [4.2014864126841225, 3.974108350245158, 3.872968837865194, 3.8136519715627033, 3.775611585553487, 3.746514253679911, 3.7268699841181436, 3.713372045389811, 3.70188271522522, 3.697400878461202]
train_ppl:  [109.59311663632973, 56.90848283930345, 46.624363255814906, 41.4752512304757, 38.1986073151506, 35.86988172960403, 34.10717735775817, 32.68353520775825, 31.517341748345107, 30.526501570387964]
val_ppl:  [66.78552815490215, 53.20265761372868, 48.08493074326038, 45.31562843531333, 43.62418007717807, 42.373122319724914, 41.54885584578678, 40.99179994622575, 40.52352684590753, 40.34231340135484]

Best Training PPL:  30.526501570387964
Best Validation PPL:  40.34231340135484


In [ ]:
SEED3 = 100
model_det3 = model_training_with_gate(train_dataset, qk_norm = True, pre_norm = True, post_norm = True, add_norm = False, remove_norm = False, vocab_size = vocab_size,
                                                                               n_transformer = 8, seed=SEED3)

train_dataloader size:  15000
val_dataloader size:  3750

--- Training with Gating ---
Epoch 0 Step 0 | Loss=11.0450 | gate_mean=0.8808 | LayerMeans=[0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881, 0.881] | LayerStd=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | LayerSparsity=[0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0] | MaxAct=3.1297 | MeanAct=0.0008 | GradNorm=3.2308 | Sparsity=0.0000 | HeadMeanAvg=0.8808 | Sink Val=0.0477 | Dead=0.000 | Open=0.000
Epoch 0 Step 500 | Loss=6.0128 | gate_mean=0.7234 | LayerMeans=[0.837, 0.693, 0.665, 0.692, 0.713, 0.719, 0.682, 0.784] | LayerStd=[0.132, 0.246, 0.29, 0.281, 0.267, 0.271, 0.283, 0.208] | LayerSparsity=[0.002, 0.016, 0.056, 0.038, 0.028, 0.036, 0.044, 0.005] | MaxAct=15.9879 | MeanAct=-3.6261 | GradNorm=0.5878 | Sparsity=0.0282 | HeadMeanAvg=0.7234 | Sink Val=0.0388 | Dead=0.001 | Open=0.015
Epoch 0 Step 1000 | Loss=5.5548 | gate_mean=0.6150 | LayerMeans=[0.819, 0.578, 0.524, 0.566, 0.606, 0.595, 0.552, 0.681] | LayerStd=[0.162, 0.303, 0.339, 0

In [ ]:
print('#### Logs for Config 3 (With Gate) - SEED 100')
print('train_loss: ', model_det3['train_loss'])
print('val_loss: ', model_det3['val_loss'])
print('train_ppl: ', model_det3['train_ppl'])
print('val_ppl: ', model_det3['val_ppl'])
print('\nBest Training PPL: ', min(model_det3['train_ppl']))
print('Best Validation PPL: ', min(model_det3['val_ppl']))

#### Logs for Config 3 (With Gate) - SEED 100
train_loss:  [4.689779433965683, 4.035689094861349, 3.8366784206231435, 3.7197213768641153, 3.637693693971634, 3.5750467362562817, 3.5246779846350353, 3.4825407517910003, 3.446429480902354, 3.414893560552597]
val_loss:  [4.202786520512899, 3.977637047068278, 3.8737398244222003, 3.814835604476929, 3.7760665137608846, 3.7469120774586995, 3.7292746772766114, 3.7158823624928794, 3.705293117904663, 3.6971534457524617]
train_ppl:  [108.82917314084797, 56.58189710194071, 46.371192754671, 41.25289849527016, 38.00408652103597, 35.69628908758628, 33.94284173965516, 32.54229903221589, 31.388120110715963, 30.413712036572548]
val_ppl:  [66.8724130105075, 53.39072528438093, 48.12201787346032, 45.36929726048638, 43.64403046212743, 42.389982708874705, 41.64888832068176, 41.09483162970876, 40.661964320030926, 40.332332628308876]

Best Training PPL:  30.413712036572548
Best Validation PPL:  40.332332628308876


In [ ]:
!nvidia-smi

In [ ]:
import pandas as pd

# Data extracted from the execution logs for Config 1 (2+), Config 2, Config 3-, and Config 3
full_results = {
    'Config': [
        'Config 1 (2+)', 'Config 1 (2+)', 'Config 1 (2+)',
        'Config 2', 'Config 2', 'Config 2',
        'Config 3-', 'Config 3-', 'Config 3-',
        'Config 3', 'Config 3', 'Config 3'
    ],
    'Seed': [12, 42, 100, 12, 42, 100, 12, 42, 100, 12, 42, 100],
    'Best Training PPL': [
        31.3446, 31.0520, 30.9635, # Config 1 (2+)
        31.8852, 31.6898, 31.5152, # Config 2
        31.0517, 31.0652, 31.0251, # Config 3-
        30.4414, 30.5265, 30.4137  # Config 3
    ],
    'Best Validation PPL': [
        40.6014, 40.4690, 40.4795, # Config 1 (2+)
        40.5446, 40.4122, 40.4897, # Config 2
        40.0113, 40.0898, 40.2814, # Config 3-
        40.0094, 40.3423, 40.3323  # Config 3
    ]
}

df_full = pd.DataFrame(full_results)

print('Consolidated Summary: Gated Transformer Experiments')
display(df_full)

# Calculate averages for clearer comparison
avg_full = df_full.groupby('Config')[['Best Training PPL', 'Best Validation PPL']].mean().reset_index()
print('\nAverage PPL per Configuration:')
display(avg_full.sort_values('Best Validation PPL'))

Consolidated Summary: Gated Transformer Experiments


,Config,Seed,Best Training PPL,Best Validation PPL
0,Config 1 (2+),12,31.3446,40.6014
1,Config 1 (2+),42,31.0520,40.4690
2,Config 1 (2+),100,30.9635,40.4795
3,Config 2,12,31.8852,40.5446
4,Config 2,42,31.6898,40.4122
5,Config 2,100,31.5152,40.4897
6,Config 3-,12,31.0517,40.0113
7,Config 3-,42,31.0652,40.0898
8,Config 3-,100,31.0251,40.2814
9,Config 3,12,30.4414,40.0094



Average PPL per Configuration:


,Config,Best Training PPL,Best Validation PPL
3,Config 3-,31.047333,40.127500
2,Config 3,30.460533,40.228000
1,Config 2,31.696733,40.482167
0,Config 1 (2+),31.120033,40.516633


In [ ]:
# from google.colab import drive

# # Mount Google Drive
# drive.mount('/content/drive')
